In [ ]:
# !pip install numba
# !pip install pymap3d

In [ ]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from eo_tools.util import show_cog
from eo_tools.S1_dev import S1IWSwath # fetch_dem_burst, geocode_burst, resample_burst_ampl
import numpy as np
from scipy.interpolate import griddata
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
data_dir = "/data/S1"
master_dir = f"{data_dir}/S1A_IW_SLC__1SDV_20230903T183344_20230903T183412_050167_0609B4_100E.SAFE"
slave_dir = f"{data_dir}/S1A_IW_SLC__1SDV_20230915T183345_20230915T183413_050342_060F9F_85A4.SAFE"
master_re = "/data/res/master_re.tiff"
slave_re = "/data/res/slave_re.tiff"
burst_idx = 5
iw = 2
pol = "vv"

In [ ]:
mst = S1IWSwath(master_dir, iw=iw, pol=pol)
slv = S1IWSwath(slave_dir, iw=iw, pol=pol)

In [ ]:
file_dem = mst.fetch_dem_burst(burst_idx)

In [ ]:
az_mst, rg_mst = mst.geocode_burst(file_dem, burst_idx=burst_idx)

In [ ]:
az_slv, rg_slv = slv.geocode_burst(file_dem, burst_idx=burst_idx)

In [ ]:
slv.resample_burst(
    file_dem,
    slave_re,
    az_slv,
    rg_slv,
    burst_idx=burst_idx,
    order=3,
    output_complex=False,
)

In [ ]:
mst.resample_burst(
    file_dem,
    master_re,
    az_mst,
    rg_mst,
    burst_idx=burst_idx,
    order=3,
    output_complex=True,
)

In [ ]:
import rasterio
with rasterio.open(master_re) as ds:
    print(ds.profile)
    toto = ds.read(1)

In [ ]:
plt.imshow(np.abs(toto), vmax=300)
plt.colorbar()

In [ ]:
from folium import Map, LayerControl
m = Map()
# show_cog(master_re, m, rescale="2,8.1", expression="log(b1+1)")
# show_cog(slave_re, m, rescale="2,8.1", expression="log(b1+1)")
show_cog(master_re, m, rescale="0,300.1")# expression="log(b1+1)")
# show_cog(slave_re, m, rescale="0,302")# expression="log(b1+1)")
LayerControl().add_to(m)
m

In [ ]:
# interpolate azimuth and range to fixed grid
from eo_tools.S1_dev import read_metadata, read_chunk

dir_tiff_mst = Path(f"{master_dir}/measurement/")
dir_xml_mst = Path(f"{master_dir}/annotation/")
pth_xml_mst = dir_xml_mst.glob(f"*iw{iw}*{pol}*.xml")
pth_tiff_mst = dir_tiff_mst.glob(f"*iw{iw}*{pol}*.tiff")
pth_xml_mst = list(pth_xml_mst)[0]
pth_tiff_mst = list(pth_tiff_mst)[0]

meta_mst = read_metadata(pth_xml_mst)


dir_tiff_slv = Path(f"{slave_dir}/measurement/")
dir_xml_slv = Path(f"{slave_dir}/annotation/")
pth_xml_slv = dir_xml_slv.glob(f"*iw{iw}*{pol}*.xml")
pth_tiff_slv = dir_tiff_slv.glob(f"*iw{iw}*{pol}*.tiff")
pth_xml_slv = list(pth_xml_slv)[0]
pth_tiff_slv = list(pth_tiff_slv)[0]

meta_slv = read_metadata(pth_xml_slv)


In [ ]:
naz = int(meta_mst["product"]["swathTiming"]["linesPerBurst"])
nrg = int(meta_mst["product"]["swathTiming"]["samplesPerBurst"])
first_line = (burst_idx - 1) * naz


naz_slv = int(meta_slv["product"]["swathTiming"]["linesPerBurst"])
nrg_slv = int(meta_slv["product"]["swathTiming"]["samplesPerBurst"])
first_line_slv = (burst_idx - 1) * naz_slv

In [ ]:
from eo_tools.S1_dev import coregister, presum
# pdb_mst = mst.deramp_burst(burst_idx)
# pdb_slv = slv.deramp_burst(burst_idx)

arr_mst = mst.read_burst(burst_idx)# * np.exp(1j*pdb_mst)
arr_slv = slv.read_burst(burst_idx)# * np.exp(1j*pdb_slv)

# az_co, rg_co, az_mst_rdr, rg_mst_rdr = coregister(arr_mst, arr_slv, az_mst, rg_mst, az_slv, rg_slv)


In [ ]:
plt.imshow(np.abs(arr_mst)[:,::16], interpolation='none')

In [ ]:
from scipy.ndimage import map_coordinates
arr_slv_re = map_coordinates(arr_slv, (az_co, rg_co), order=3).reshape(*arr_mst.shape) 


In [ ]:
pht_mst = mst.phi_topo(rg_mst_rdr).reshape(*arr_mst.shape)
pht_slv = slv.phi_topo(rg_co).reshape(*arr_mst.shape)
# plt.imshow(np.angle(np.exp(1j*(pht_mst-pht_slv)))[800:], interpolation="none")

In [ ]:
phi2 = np.angle(presum(arr_slv_re*arr_mst.conj(), 2, 16))
phi3 = np.angle(presum(arr_slv_re*arr_mst.conj()*np.exp(-1j*(pht_mst-pht_slv)), 2, 16))

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(phi2[:,800:], cmap="hsv", interpolation="none")
# plt.figure(figsize=(20,20))
# plt.imshow(im_mst[:,::8], vmin=-np.pi, vmax=np.pi)

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(phi3[:,800:], cmap="hsv", interpolation="none")
# plt.figure(figsize=(20,20))
# plt.imshow(im_mst[:,::8], vmin=-np.pi, vmax=np.pi)

In [ ]:
meta_burst = mst.meta["product"]["swathTiming"]["burstList"]["burst"][burst_idx - 1]

In [ ]:
nlines = int(meta_burst["firstValidSample"]["@count"])
arr = mst.read_burst(1).reshape(nlines, -1)
# first_sample_list 
fs_str = meta_burst["firstValidSample"]["#text"]
ls_str = meta_burst["lastValidSample"]["#text"]
first_sample_arr = np.array((fs_str).split(" "), dtype="float64")
last_sample_arr = np.array((ls_str).split(" "), dtype="float64")

In [ ]:
arr.shape